Samuel Scott - Module 2 Homework - Takyo Software Cataloger

In [1]:
#!pip install mlxtend
#!pip install xgboost

In [23]:
#Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as sm
from mlxtend.feature_selection import ExhaustiveFeatureSelector, SequentialFeatureSelector
from sklearn.linear_model import BayesianRidge, Lasso, LassoCV, LinearRegression, Ridge, RidgeCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import scipy
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc, roc_curve
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
import warnings
warnings.filterwarnings("ignore")

## Data Sanity Check

In [3]:
tayko_data = pd.read_csv('C:/Users/samsc/Desktop/ADS-505/Tayko.csv')
tayko_data.head()

,sequence_number,US,source_a,source_c,source_b,source_d,source_e,source_m,source_o,source_h,...,source_x,source_w,Freq,last_update_days_ago,1st_update_days_ago,Web order,Gender=male,Address_is_res,Purchase,Spending
0,1,1,0,0,1,0,0,0,0,0,...,0,0,2,3662,3662,1,0,1,1,128
1,2,1,0,0,0,0,1,0,0,0,...,0,0,0,2900,2900,1,1,0,0,0
2,3,1,0,0,0,0,0,0,0,0,...,0,0,2,3883,3914,0,0,0,1,127
3,4,1,0,1,0,0,0,0,0,0,...,0,0,1,829,829,0,1,0,0,0
4,5,1,0,1,0,0,0,0,0,0,...,0,0,1,869,869,0,0,0,0,0


In [4]:
tayko_data.shape

(2000, 25)

In [5]:
#No missing data
tayko_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   sequence_number       2000 non-null   int64
 1   US                    2000 non-null   int64
 2   source_a              2000 non-null   int64
 3   source_c              2000 non-null   int64
 4   source_b              2000 non-null   int64
 5   source_d              2000 non-null   int64
 6   source_e              2000 non-null   int64
 7   source_m              2000 non-null   int64
 8   source_o              2000 non-null   int64
 9   source_h              2000 non-null   int64
 10  source_r              2000 non-null   int64
 11  source_s              2000 non-null   int64
 12  source_t              2000 non-null   int64
 13  source_u              2000 non-null   int64
 14  source_p              2000 non-null   int64
 15  source_x              2000 non-null   int64
 16  source

## The Company

Tayko is a software catalog firm that sells games and educational software. It started as a software
manufacturer and later expanded its offerings by adding third-party titles. Tayko has recently assembled a
revised collection of items in a new catalog, which it is preparing to roll out in a large mailing campaign.
Tayko's customer list is a key strategic asset. To expand its customer base, Tayko has joined a
consortium of catalog firms that specialize in computer and software products. Consortium members pool
their customer lists and can withdraw an equivalent number of names each quarter for their own mailings.
Members are allowed to apply predictive modeling to the pooled records so they can select names more
effectively, mailing to those most likely to respond rather than mailing to the list at random

## The Mailing Experiment

Tayko supplied its 200,000-name customer list to the consortium pool, which now totals over 5,000,000
names. This entitles Tayko to draw 200,000 names for an upcoming mailing. To improve the odds of
selecting high-performing prospects, Tayko first ran a test by drawing 20,000 names from the pool and
mailing the new catalog to that test sample.
The test mailing produced 1,065 purchasers, a response rate of 5.3%. To improve the signal available to
the modeling techniques, the working dataset was constructed as a stratified sample with equal numbers
of purchasers and non-purchasers (1,000 each), producing an apparent response rate of 50%. After
modeling, the predicted probabilities must be adjusted back to the true population rate by multiplying each
case's probability of purchase by 0.053 / 0.5 = 0.107.

### Question 1 - Baseline Profit Estimate
Each catalog costs approximately $2 to mail (including printing, postage, and mailing costs). Estimate the
gross profit that Tayko could expect from the remaining 180,000 names if it selects them randomly from
the pool. This becomes your business baseline. Every model later in the assignment must be compared
against it

In [6]:
#Baseline model
num_of_names = 180000
cost = num_of_names * 2 # $360,000 cost
response_rate = 0.053 #5.3% in decimal form
avg_spender = tayko_data['Spending'].mean().round(2) #average customer spends $102.62
earnings = (num_of_names * response_rate) * avg_spender #the earnings are the customers guaranteed to respond times the amount the avg_spender spends
gross_profit = earnings - cost 
gross_profit

np.float64(618994.8)

The baseline model projects the gross profit to be $618,994.8

### Question 2 - Classification Model: Purchaser vs. Non-Purchaser
Develop a model for classifying a customer as a purchaser or non-purchaser.

#### 2.1 Partition the Data
Partition the data randomly into a training set (800 records), validation set (700 records), and test set (500
records). Use a fixed random seed for reproducibility.

In [7]:
#Partition training vs test
outcome = 'Purchase'
predictors = tayko_data.drop(columns=['sequence_number', 'Spending', 'Purchase'])
predictors = predictors.columns

X = tayko_data[predictors]
y = tayko_data[outcome]

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=1)

print(train_X.shape)
print(test_X.shape)
print(train_y.shape)
print(test_y.shape)

(1500, 22)
(500, 22)
(1500,)
(500,)


In [9]:
#Partition training vs holdout
train_X, holdout_X, train_y, holdout_y = train_test_split(train_X, train_y, test_size=700/1500, random_state=1)
print(train_X.shape)
print(holdout_X.shape)
print(train_y.shape)
print(holdout_y.shape)
print(test_X.shape)
print(test_y.shape)

(426, 22)
(374, 22)
(426,)
(374,)
(500, 22)
(500,)


#### 2.2 Logistic Regression with L2 Penalty
Using only the **training set**, run a cross-validated logistic regression with an L2 penalty using
LogisticRegressionCV with parameters solver='lbfgs', cv=5, and max_iter=500. Use this
model to classify the data into purchasers and non-purchasers. Logistic regression is chosen here
because it yields an estimated probability of purchase, which is required later when you compute
expected spending.

In [24]:
log_reg = LogisticRegressionCV(cv=5, max_iter=500, solver='lbfgs', l1_ratios=None, scoring=None, use_legacy_attributes=True, random_state=1)
log_reg.fit(train_X, train_y)

,"l1_ratios l1_ratios: array-like of shape (n_l1_ratios), default=NoneFloats between 0 and 1 passed as Elastic-Net mixing parameter (scaling betweenL1 and L2 penalties). For `l1_ratio = 0` the penalty is an L2 penalty. For`l1_ratio = 1` it is an L1 penalty. For `0 < l1_ratio < 1`, the penalty is acombination of L1 and L2.All the values of the given array-like are tested by cross-validation and theone giving the best prediction score is used... warning:: Certain values of `l1_ratios`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `l1_ratios=None` is deprecated in 1.8 and will raise an error in version 1.10. Default value will change from `None` to `(0.0,)` in version 1.10.",None
,"cv cv: int or cross-validation generator, default=NoneThe default cross-validation generator used is Stratified K-Folds.If an integer is provided, it specifies the number of folds, `n_folds`, used.See the module :mod:`sklearn.model_selection` module for thelist of possible cross-validation objects... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str or callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: :ref:`accuracy <accuracy_score>` is used... versionchanged:: 1.11 The default will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11.",None
,"max_iter max_iter: int, default=100Maximum number of iterations of the optimization algorithm.",500
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.Note that this only applies to the solver and not the cross-validationgenerator. See :term:`Glossary <random_state>` for details.",1
,"use_legacy_attributes use_legacy_attributes: bool, default=TrueIf True, use legacy values for attributes:- `C_` is an ndarray of shape (n_classes,) with the same value repeated- `l1_ratio_` is an ndarray of shape (n_classes,) with the same value repeated- `coefs_paths_` is a dict with class labels as keys and ndarrays as values- `scores_` is a dict with class labels as keys and ndarrays as values- `n_iter_` is an ndarray of shape (1, n_folds, n_cs) or similarIf False, use new values for attributes:- `C_` is a float- `l1_ratio_` is a float- `coefs_paths_` is an ndarray of shape (n_folds, n_l1_ratios, n_cs, n_classes, n_features) For binary problems (n_classes=2), the 2nd last dimension is 1.- `scores_` is an ndarray of shape (n_folds, n_l1_ratios, n_cs)- `n_iter_` is an ndarray of shape (n_folds, n_l1_ratios, n_cs).. versionchanged:: 1.10 The default will change from True to False in version 1.10... deprecated:: 1.10 `use_legacy_attributes` will be deprecated in version 1.10 and be removed in 1.12.",True
,"Cs Cs: int or list of floats, default=10Each of the values in Cs describes the inverse of regularizationstrength. If Cs is as an int, then a grid of Cs values are chosenin a logarithmic scale between 1e-4 and 1e4.Like in support vector machines, smaller values specify strongerregularization.",10
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer dual=False whenn_samples > n_features.",False
,"penalty penalty: {'l1', 'l2', 'elasticnet'}, default='l2'Specify the norm of the penalty:- `'l2'`: add an L2 penalty term (used by default);- `'l1'`: add an L1 penalty term;- `'elasticnet'

In [29]:
#Making sure 
print(log_reg.classes_)
print(log_reg.coef_) #a coefficient for each predictor
print(log_reg.intercept_)
print(log_reg.scores_) #CV scores for each C score which defaulted to 10

[0 1]
[[ 1.66965157e-01  7.74442812e-01 -5.31875167e-01 -2.47914901e-01
   2.16731457e-01 -1.52012270e-01  6.67120435e-02 -8.37667581e-02
  -1.60681041e+00  1.37787994e-01  9.56123281e-02  8.73125047e-02
   8.63607938e-01  3.82616976e-01 -2.00826944e-01  6.73572229e-01
   1.87535268e+00  2.21397299e-04 -1.93135428e-04  7.51622908e-01
  -1.69057503e-01 -9.29732016e-01]]
[-2.72539899]
{np.int64(1): array([[0.6627907 , 0.6627907 , 0.77906977, 0.8255814 , 0.79069767,
        0.77906977, 0.77906977, 0.77906977, 0.77906977, 0.77906977],
       [0.6       , 0.61176471, 0.72941176, 0.83529412, 0.88235294,
        0.83529412, 0.82352941, 0.82352941, 0.82352941, 0.82352941],
       [0.69411765, 0.70588235, 0.70588235, 0.70588235, 0.75294118,
        0.76470588, 0.76470588, 0.77647059, 0.77647059, 0.77647059],
       [0.61176471, 0.61176471, 0.64705882, 0.69411765, 0.78823529,
        0.8       , 0.8       , 0.8       , 0.8       , 0.8       ],
       [0.62352941, 0.62352941, 0.69411765, 0.788235